# EDA (Exploratory Data Analysis) of the Abalone dataset

Goal: understand the data before predicting the age of an abalone (number of `Rings`) from its physical measurements.

Dataset: [Kaggle - Abalone](https://www.kaggle.com/datasets/rodolfomendes/abalone-dataset), saved as `data/abalone.csv`.

# Imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 500)

# Data

In [ ]:
DATA_PATH = "../data/abalone.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicated rows: {df.duplicated().sum()}")

**Conclusion:** 4,177 abalones, 8 features (1 categorical: `Sex`, 7 numerical measurements) and the target `Rings`.
No missing values and no duplicates, so no imputation is needed.

# Target: `Rings`

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df["Rings"], bins=29, ax=ax)
ax.set_title("Distribution of Rings")
plt.show()

print(f"Skewness: {df['Rings'].skew():.2f}")

**Conclusion:** `Rings` goes from 1 to 29, with most abalones between 8 and 11 rings (mean ≈ 9.9).
The distribution is right-skewed (skewness ≈ 1.1): old abalones are rare, so the model will probably be less accurate on them.
The age in years is `Rings + 1.5`; we predict `Rings` directly.

# Categorical feature: `Sex`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Sex", ax=axes[0])
axes[0].set_title("Number of abalones by Sex")
sns.boxplot(data=df, x="Sex", y="Rings", ax=axes[1])
axes[1].set_title("Rings by Sex")
plt.show()

df.groupby("Sex")["Rings"].agg(["count", "mean", "median"])

**Conclusion:** the 3 classes are balanced (M: 1,528, I: 1,342, F: 1,307).
Infants (`I`) have clearly fewer rings (mean ≈ 7.9) than females (≈ 11.1) and males (≈ 10.7), so `Sex` is useful and must be kept as a one-hot encoded feature.

# Numerical features

In [ ]:
NUMERICAL_COLS = ["Length", "Diameter", "Height", "Whole weight", "Shucked weight", "Viscera weight", "Shell weight"]

df[NUMERICAL_COLS].hist(bins=40, figsize=(14, 8))
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[NUMERICAL_COLS + ["Rings"]].corr(), annot=True, fmt=".2f", cmap="Blues", ax=ax)
ax.set_title("Correlation matrix")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x="Shell weight", y="Rings", hue="Sex", alpha=0.5, ax=ax)
ax.set_title("Rings vs Shell weight")
plt.show()

**Conclusion:** all measurements are positively correlated with `Rings` (from 0.42 for `Shucked weight` to 0.63 for `Shell weight`).
The features are also very correlated with each other (0.78 to 0.99): size and weight grow together.
A linear model can still predict well, but its coefficients should not be interpreted one by one.
The relation with `Rings` is not perfectly linear (more spread for big abalones).

# Outliers

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
sns.boxplot(data=df, x="Height", ax=ax)
ax.set_title("Height")
plt.show()

df[(df["Height"] == 0) | (df["Height"] > 0.4)]

**Conclusion:** 2 abalones have `Height == 0`, which is physically impossible: they are data errors and will be removed.
2 abalones have a very large height (> 0.4) compared to their other measurements; there are only 2 of them, so we keep them.

# Final conclusions

- Clean dataset: no missing values, no duplicates. Only 2 rows to remove (`Height == 0`).
- Target `Rings` is right-skewed: old abalones are rare and will be harder to predict.
- `Sex` matters (infants are younger) → one-hot encode it.
- All 7 measurements are correlated with `Rings`, `Shell weight` the most, and strongly correlated with each other.
- Next step (`modelling.ipynb`): use all features with a simple linear regression as a first model.